In [2]:
# 01 - Data Loading and Understanding

"""This notebook loads the transformed e-commerce price data from BigQuery and performs a first structural understanding of the dataset.

Main objectives:
- Load analytics tables from BigQuery
- Inspect dataset size and schema
- Identify the available time period
- Count products, sites, and categories
- Prepare dataframes for further statistical analysis"""

'This notebook loads the transformed e-commerce price data from BigQuery and performs a first structural understanding of the dataset.\n\nMain objectives:\n- Load analytics tables from BigQuery\n- Inspect dataset size and schema\n- Identify the available time period\n- Count products, sites, and categories\n- Prepare dataframes for further statistical analysis'

In [3]:
import pandas as pd
import numpy as np

from google.cloud import bigquery

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

In [4]:
#Connexion BigQuery
PROJECT_ID = "price-intel-prod"
DATASET_ID = "price_staging"

client = bigquery.Client(project=PROJECT_ID)


c:\Users\Usuario\Desktop\analytics\venv\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [6]:
query_stg = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET_ID}.stg_prices`
"""

query_clean = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET_ID}.clean_prices`
"""


query_agg = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET_ID}.agg_prices`
"""

query_ts = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET_ID}.price_timeseries`
"""

query_intelligence = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET_ID}.price_intelligence`
"""
df_stg = client.query(query_stg).to_dataframe()
df_clean = client.query(query_clean).to_dataframe()
df_agg = client.query(query_agg).to_dataframe()
df_ts = client.query(query_ts).to_dataframe()
df_intelligence = client.query(query_intelligence).to_dataframe()



c:\Users\Usuario\Desktop\analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [10]:
tables_shapes = {
    "stg_prices": df_stg.shape,
    "clean_prices": df_clean.shape,
    "agg_prices": df_agg.shape,
    "price_time_series": df_ts.shape,
    "price_intelligence": df_intelligence.shape,
}

tables_shapes

print (tables_shapes)

"""Although the main statistical analysis will rely mostly on `clean_prices`, `price_time_series`, and `price_intelligence`, the staging and aggregated tables are also loaded for validation and comparison purposes."""

{'stg_prices': (10526, 13), 'clean_prices': (10523, 13), 'agg_prices': (934, 11), 'price_time_series': (10523, 12), 'price_intelligence': (934, 14)}


'Although the main statistical analysis will rely mostly on `clean_prices`, `price_time_series`, and `price_intelligence`, the staging and aggregated tables are also loaded for validation and comparison purposes.'

In [13]:
#Data Overview

In [14]:
df_clean.head()

,product_id,site_name,site_product_id,product_name,price,currency,availability,category,image_url,source_url,rating,review_count,scraped_at
0,ELECTRO-35BBDD1F,electroplanet,ELECTRO-35BBDD1F,ADAPTATEUR USB 3.0 - ETHERNET GIGABIT RJ45 TPLINK,299.0,MAD,in_stock,adaptateurs,https://prod2-media.electroplanet.ma/media/cat...,https://www.electroplanet.ma/p2822704-tp-link-...,NaN,<NA>,2026-05-02 11:59:10.666404+00:00
1,ELECTRO-35BBDD1F,electroplanet,ELECTRO-35BBDD1F,ADAPTATEUR USB 3.0 - ETHERNET GIGABIT RJ45 TPLINK,299.0,MAD,in_stock,adaptateurs,https://prod2-media.electroplanet.ma/media/cat...,https://www.electroplanet.ma/p2822704-tp-link-...,NaN,<NA>,2026-05-02 13:45:35.227456+00:00
2,ELECTRO-35BBDD1F,electroplanet,ELECTRO-35BBDD1F,ADAPTATEUR USB 3.0 - ETHERNET GIGABIT RJ45 TPLINK,299.0,MAD,in_stock,adaptateurs,https://prod2-media.electroplanet.ma/media/cat...,https://www.electroplanet.ma/p2822704-tp-link-...,NaN,<NA>,2026-05-03 16:21:43.267692+00:00
3,ELECTRO-35BBDD1F,electroplanet,ELECTRO-35BBDD1F,ADAPTATEUR USB 3.0 - ETHERNET GIGABIT RJ45 TPLINK,299.0,MAD,in_stock,adaptateurs,https://prod2-media.electroplanet.ma/media/cat...,https://www.electroplanet.ma/p2822704-tp-link-...,NaN,<NA>,2026-05-01 23:51:36.022527+00:00
4,ELECTRO-35BBDD1F,electroplanet,ELECTRO-35BBDD1F,ADAPTATEUR USB 3.0 - ETHERNET GIGABIT RJ45 TPLINK,299.0,MAD,in_stock,adaptateurs,https://prod2-media.electroplanet.ma/media/cat...,https://www.electroplanet.ma/p2822704-tp-link-...,NaN,<NA>,2026-05-02 12:56:36.019534+00:00


In [15]:
df_ts.head()

,product_id,site_name,product_name,category,price,previous_price,availability,scraped_at,price_change_pct,price_rolling_avg,price_rolling_min,price_rolling_max
0,ELECTRO-01EB89FF,electroplanet,"LS49FG910EUXEN 49""MONITEURODYSSEY G9 G91F SAMSUNG",ecran,11999.0,NaN,in_stock,2026-05-02 00:12:58.795706+00:00,0.0,11999.0,11999.0,11999.0
1,ELECTRO-04677085,electroplanet,MSI-THIN-B12VE 15.6 I7-13620H 16GB 512G RTX4050,pc-gamer,15999.0,NaN,in_stock,2026-05-02 00:12:36.671610+00:00,0.0,15999.0,15999.0,15999.0
2,ELECTRO-04E8FF5F,electroplanet,TABLETTE MATEPAD T8 LTE 2GB+32GB BLEU HUAWEI,tablettes-android,1590.0,NaN,in_stock,2026-05-02 00:12:08.673198+00:00,0.0,1590.0,1590.0,1590.0
3,ELECTRO-04E8FF5F,electroplanet,TABLETTE MATEPAD T8 LTE 2GB+32GB BLEU HUAWEI,tablettes-android,1590.0,1590.0,in_stock,2026-05-02 11:58:39.805042+00:00,0.0,1590.0,1590.0,1590.0
4,ELECTRO-058A8739,electroplanet,"VIVOBOOK I9-13900H 31E 16-1T SSD W11 15"" SLV ASUS",notebook,12999.0,NaN,in_stock,2026-05-01 23:51:19.988432+00:00,0.0,12999.0,12999.0,12999.0


In [16]:
df_intelligence.head()

,product_id,site_name,product_name,category,current_price,current_availability,price_min,price_max,price_avg,price_change_pct,source_url,image_url,first_scraped_at,last_scraped_at
0,JUMIA-A4FDE41D,jumia_ma,"Support TV mural pour LCD 14-42 pouces, capaci...",electronique-accessoires-fournitures,23.0,in_stock,23.0,24.0,23.520000,-4.17,https://www.jumia.ma/generic-support-tv-mural-...,https://ma.jumia.is/unsafe/fit-in/300x300/filt...,2026-05-01 23:34:47.517623+00:00,2026-05-26 00:22:21.028470+00:00
1,JUMIA-2724A507,jumia_ma,Télécommande universelle pour Samsung Smart TV...,electronique-accessoires-fournitures,31.0,in_stock,31.0,32.0,31.882353,-3.13,https://www.jumia.ma/generic-telecommande-univ...,https://ma.jumia.is/unsafe/fit-in/300x300/filt...,2026-05-01 23:34:47.504828+00:00,2026-06-10 13:47:11.681115+00:00
2,JUMIA-C701395F,jumia_ma,Support de téléphone magnétique pliable rotati...,accessoires-telephone,33.0,in_stock,33.0,39.0,35.733333,-15.38,https://www.jumia.ma/generic-support-de-teleph...,https://ma.jumia.is/unsafe/fit-in/300x300/filt...,2026-05-01 23:34:41.997310+00:00,2026-06-10 13:47:02.663716+00:00
3,JUMIA-2B8F176B,jumia_ma,Cable Adaptateur jack 3.5 mm Audio Mic Microph...,electronique-accessoires-fournitures,33.0,in_stock,30.0,33.0,32.769231,10.00,https://www.jumia.ma/generic-cable-adaptateur-...,https://ma.jumia.is/unsafe/fit-in/300x300/filt...,2026-05-04 23:54:33.316995+00:00,2026-06-10 13:47:11.704373+00:00
4,JUMIA-56AD8C0C,jumia_ma,Support Telephone Portable Pour PC Portable Or...,accessoires-telephone,35.0,in_stock,35.0,39.0,35.923077,-10.26,https://www.jumia.ma/support-telephone-portabl...,https://ma.jumia.is/unsafe/fit-in/300x300/filt...,2026-05-25 09:58:22.381853+00:00,2026-05-26 00:22:12.709165+00:00


In [17]:
#Columns types

In [18]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 10523 entries, 0 to 10522
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   product_id       10523 non-null  str                
 1   site_name        10523 non-null  str                
 2   site_product_id  10523 non-null  str                
 3   product_name     10523 non-null  str                
 4   price            10523 non-null  float64            
 5   currency         10523 non-null  str                
 6   availability     10523 non-null  str                
 7   category         10523 non-null  str                
 8   image_url        10523 non-null  str                
 9   source_url       10523 non-null  str                
 10  rating           4878 non-null   float64            
 11  review_count     3523 non-null   Int64              
 12  scraped_at       10523 non-null  datetime64[us, UTC]
dtypes: Int64(1), datetime64[us,

In [19]:
df_ts.info()

<class 'pandas.DataFrame'>
RangeIndex: 10523 entries, 0 to 10522
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   product_id         10523 non-null  str                
 1   site_name          10523 non-null  str                
 2   product_name       10523 non-null  str                
 3   category           10523 non-null  str                
 4   price              10523 non-null  float64            
 5   previous_price     9589 non-null   float64            
 6   availability       10523 non-null  str                
 7   scraped_at         10523 non-null  datetime64[us, UTC]
 8   price_change_pct   10523 non-null  float64            
 9   price_rolling_avg  10523 non-null  float64            
 10  price_rolling_min  10523 non-null  float64            
 11  price_rolling_max  10523 non-null  float64            
dtypes: datetime64[us, UTC](1), float64(6), str(5)
memory usag

In [20]:
df_intelligence.info()

<class 'pandas.DataFrame'>
RangeIndex: 934 entries, 0 to 933
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype              
---  ------                --------------  -----              
 0   product_id            934 non-null    str                
 1   site_name             934 non-null    str                
 2   product_name          934 non-null    str                
 3   category              934 non-null    str                
 4   current_price         934 non-null    float64            
 5   current_availability  934 non-null    str                
 6   price_min             934 non-null    float64            
 7   price_max             934 non-null    float64            
 8   price_avg             934 non-null    float64            
 9   price_change_pct      934 non-null    float64            
 10  source_url            934 non-null    str                
 11  image_url             934 non-null    str                
 12  first_scraped_at   

In [21]:
#Dates conversion

df_clean["scraped_at"] = pd.to_datetime(df_clean["scraped_at"])
df_ts["scraped_at"] = pd.to_datetime(df_ts["scraped_at"])

df_intelligence["first_scraped_at"] = pd.to_datetime(df_intelligence["first_scraped_at"])
df_intelligence["last_scraped_at"] = pd.to_datetime(df_intelligence["last_scraped_at"])

In [22]:
#Summary

summary = {
    "clean_rows": len(df_clean),
    "time_series_rows": len(df_ts),
    "intelligence_rows": len(df_intelligence),
    "unique_products": df_clean["product_id"].nunique(),
    "unique_sites": df_clean["site_name"].nunique(),
    "unique_categories": df_clean["category"].nunique(),
    "first_scraping_date": df_clean["scraped_at"].min(),
    "last_scraping_date": df_clean["scraped_at"].max()
}

summary

{'clean_rows': 10523,
 'time_series_rows': 10523,
 'intelligence_rows': 934,
 'unique_products': 934,
 'unique_sites': 2,
 'unique_categories': 18,
 'first_scraping_date': Timestamp('2026-05-01 23:34:41.976577+0000', tz='UTC'),
 'last_scraping_date': Timestamp('2026-06-10 14:00:48.028621+0000', tz='UTC')}

In [23]:
# Markdown

"""The dataset contains transformed e-commerce price data collected from two websites: Jumia Morocco and Electroplanet.

The `clean_prices` table contains the cleaned product-level observations, while `price_time_series` stores historical price observations for temporal analysis. The `price_intelligence` table provides the latest consolidated view of each product, including current price, price range, and availability.

The data covers the period from 2026-05-01 to 2026-06-10, with 18 product categories and 2 e-commerce sites. This structure is suitable for both descriptive statistics and inferential statistical testing."""

'The dataset contains transformed e-commerce price data collected from two websites: Jumia Morocco and Electroplanet.\n\nThe `clean_prices` table contains the cleaned product-level observations, while `price_time_series` stores historical price observations for temporal analysis. The `price_intelligence` table provides the latest consolidated view of each product, including current price, price range, and availability.\n\nThe data covers the period from 2026-05-01 to 2026-06-10, with 18 product categories and 2 e-commerce sites. This structure is suitable for both descriptive statistics and inferential statistical testing.'